# UNIVERSAL LOAD

In [ ]:
# ==== IMPORTS ====
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import norm
import calendar
import re

# ==== SETTINGS ====
EXCLUDE_2026 = False # Whether to exclude 2026 data (which is incomplete) from the analysis. Set to False to include it.

# ---- IMPORT SETTINGS ----
IMPORT_PATH = "../../data/gcp_manual_copy/"

# ---- FILES ----
# Metrics files
FILE_ALL = "thesis_meta_all_metrics_except_grade_27032026.parquet"
FILE_FILTERED_GRADES = "thesis_meta_all_metrics_with_86pct_grades_09042026.parquet"

# ==== EXPORT SETTINGS ====
EXPORT = True # Whether to export the enriched DataFrame after analysis
EXPORT_PATH = "../../data/data_analysis_files/"

In [15]:
# ==== FUNCTIONS ====
def load_csv_to_df(csv_path, sep=";"):
    try:
        df = pd.read_csv(csv_path, encoding="utf-8", sep=sep)
        print(f"Successfully loaded CSV from {csv_path}")
        print(f"DataFrame shape: {df.shape}")
        print(f"DataFrame columns: {df.columns.tolist()}\n")
        return df
    except Exception as e:
        print(f"Error loading CSV from {csv_path}: {e}")
        return None

def load_parquet_to_df(parquet_path, na=False):
    try:
        df = pd.read_parquet(parquet_path)
        print(f"Successfully loaded Parquet from {parquet_path}")
        print(f"DataFrame shape: {df.shape}")
        if na:
            print(f"DataFrame N/A counts:\n{df.isna().sum()}\n")
        print(f"DataFrame columns: {df.columns.tolist()}\n")
        return df
    except Exception as e:
        print(f"Error loading Parquet from {parquet_path}: {e}")
        return None

In [16]:
# ==== LOAD DATAFRAMES ====
df_all = load_parquet_to_df(IMPORT_PATH + FILE_ALL)
df_filtered = load_parquet_to_df(IMPORT_PATH + FILE_FILTERED_GRADES)

# ==== DROP NA COLUMNS ====
df_all_noNA = df_all.dropna(axis=1, how="all")
print(f"df_all_noNA shape: {df_all_noNA.shape}")
print(f"df_all_noNA columns: {df_all_noNA.columns.tolist()}\n")
df_filtered_noNA = df_filtered.dropna(axis=1, how="all")
print(f"df_filtered_noNA shape: {df_filtered_noNA.shape}")
print(f"df_filtered_noNA columns: {df_filtered_noNA.columns.tolist()}\n")

# ==== COLUMNS TO DROP ====
drop_columns = [
    "access_ss",
    "Affiliations",
    "collection_facet",
    "format",
    "fulltext_availability_facet",
    "ISBN",
    "Journal Page",
    "isolanguage_facet",
    "Publisher",
    "Source",
    "source_all_ss",
    "match_trigger",
    "equation_pipeline_version",
    "pdf_file_analysis",
    "num_tot_pages_analysis",
    "num_cont_pages_analysis",
    "num_words_full_analysis",
    "num_words_cont_analysis",
    "abstract_ts_analysis",
    "Author_analysis",
    "Publication Year_analysis",
    "primary_member_id_s_analysis",
    "Title_analysis",
    ]

# ==== DROP COLUMNS ====
df_all_noNA_clean = df_all_noNA.drop(columns=drop_columns, errors="ignore")
print(f"df_all_noNA_clean shape: {df_all_noNA_clean.shape}")
print(f"df_all_noNA_clean columns: {df_all_noNA_clean.columns.tolist()}\n")
df_filtered_noNA_clean = df_filtered_noNA.drop(columns=drop_columns, errors="ignore")
print(f"df_filtered_noNA_clean shape: {df_filtered_noNA_clean.shape}")
print(f"df_filtered_noNA_clean columns: {df_filtered_noNA_clean.columns.tolist()}\n")

# ==== LOCK DATAFRAMES FOR ANALYSIS ====
print("Final DataFrames for Analysis:")
df_all_final = df_all_noNA_clean.copy()
print(f"df_all_final shape: {df_all_final.shape}")
df_filtered_final = df_filtered_noNA_clean.copy()
print(f"df_filtered_final shape: {df_filtered_final.shape}")

# ==========================================
# FEATURE ENGINEERING
# ==========================================
# A. Number of Authors
# Count semicolons using the native pandas string accessor.
# If the value is missing (NaN), fillna(0) treats it as 0 semicolons, resulting in 1 author.
df_all_final['num_authors'] = df_all_final['BY'].str.count(';').fillna(0) + 1
df_filtered_final['num_authors'] = df_filtered_final['BY'].str.count(';').fillna(0) + 1

# B. Handin Month to Number (1-12)
month_map = {v.lower(): k for k, v in enumerate(calendar.month_abbr) if k > 0}
month_map.update({v.lower(): k for k, v in enumerate(calendar.month_name) if k > 0})

def get_month_num(text):
    if pd.isna(text): return np.nan
    clean_text = re.sub(r'\d+', '', str(text)).strip().lower()
    # Try first 3 letters
    abbr = clean_text[:3]
    if abbr in month_map: return month_map[abbr]
    return np.nan

df_filtered_final['handin_month_num'] = df_filtered_final['handin_month'].apply(get_month_num)
df_all_final['handin_month_num'] = df_all_final['handin_month'].apply(get_month_num)

Successfully loaded Parquet from ../../data/gcp_manual_copy/thesis_meta_all_metrics_except_grade_27032026.parquet
DataFrame shape: (19690, 76)
DataFrame columns: ['abstract_ts', 'access_ss', 'Affiliations', 'Timestamp', 'Author', 'citation_count_i', 'ID', 'dtu_library_collection_facet', 'collection_facet', 'Publication Year', 'Conference', 'DOI', 'Editor', 'embargo_ssf', 'format', 'fulltext_availability_facet', 'has_openaccess_fulltext_b', 'holdings_ssf', 'ISBN', 'Journal Issue', 'journal_issue_tsort', 'journal_oa_model_ss', 'Journal Page', 'journal_page_start_tsort', 'Journal Title', 'journal_title_facet', 'toc_key_s', 'Journal Volume', 'journal_vol_tsort', 'keywords_ts', 'keywords_facet', 'keywords_normalized', 'isolanguage_facet', 'member_id_ss', 'ORCID', 'primary_member_id_s', 'Publisher', 'Source', 'source_all_ss', 'Title', 'MASTER THESIS TITLE', 'BY', 'SUPERVISED BY', 'YEAR', 'PUBLISHER', 'TYPES', 'pdf_file', 'num_tot_pages', 'num_cont_pages', 'num_words_full', 'num_words_cont', 

In [17]:
if EXCLUDE_2026:
    # drop rows that has "Publication Year" == 2026
    print("df_all_final")
    df_all_final = df_all_final[df_all_final["Publication Year"] != 2026]
    excluded_rows_all = pd.to_numeric(df_all_noNA_clean["Publication Year"], errors="coerce").eq(2026).sum()
    print(f"Number of rows dropped with Publication Year == 2026: {excluded_rows_all}")
    print(f"df_all_final shape after dropping rows with Publication Year == 2026: {df_all_final.shape}")
    print(f"df_all_final has columns: {df_all_final.columns.tolist()}\n")

    print("df_filtered_final")
    df_filtered_final = df_filtered_final[df_filtered_final["Publication Year"] != 2026]
    excluded_rows_filtered = pd.to_numeric(df_filtered_noNA_clean["Publication Year"], errors="coerce").eq(2026).sum()
    print(f"Number of rows dropped with Publication Year == 2026: {excluded_rows_filtered}")
    print(f"df_filtered_final shape after dropping rows with Publication Year == 2026: {df_filtered_final.shape}")
    print(f"df_filtered_final has columns: {df_filtered_final.columns.tolist()}\n")
else:
    print("Keeping rows with Publication Year == 2026.")
    

df_all_final
Number of rows dropped with Publication Year == 2026: 6
df_all_final shape after dropping rows with Publication Year == 2026: (19684, 34)
df_all_final has columns: ['abstract_ts', 'Timestamp', 'Author', 'ID', 'Publication Year', 'keywords_ts', 'keywords_facet', 'keywords_normalized', 'member_id_ss', 'primary_member_id_s', 'Title', 'MASTER THESIS TITLE', 'BY', 'SUPERVISED BY', 'pdf_file', 'num_tot_pages', 'num_cont_pages', 'num_words_full', 'num_words_cont', 'handin_month', 'num_figures', 'num_tables', 'num_references', 'equation_count', 'pdf_sha256', 'total_sentences', 'total_words', 'unique_words', 'avg_sentence_length', 'avg_word_length', 'lexical_diversity', 'Department_new', 'num_authors', 'handin_month_num']

df_filtered_final
Number of rows dropped with Publication Year == 2026: 3
df_filtered_final shape after dropping rows with Publication Year == 2026: (6251, 47)
df_filtered_final has columns: ['Timestamp', 'Author', 'ID', 'Publication Year', 'member_id_ss', 'prima

In [18]:
matched_rows = df_filtered_final["grading_total_score"].notna().sum()
total_rows = len(df_filtered_final)

pct_grades = round(matched_rows/total_rows * 100)
date_stamp = pd.Timestamp.now().strftime("%d%m%Y")

# ==== EXPORT ENRICHED DATAFRAME ====

if EXPORT:
    if EXCLUDE_2026:
        export_name_filtered = f"df_filtered_final_{pct_grades}pct_grades_excl_2026_{date_stamp}.parquet"
        df_filtered_final.to_parquet(EXPORT_PATH + export_name_filtered, index=False)
        print(f"Exported enriched DataFrame to {EXPORT_PATH + export_name_filtered}")

        export_name_all = f"df_all_final_{pct_grades}pct_grades_excl_2026_{date_stamp}.parquet"
        df_all_final.to_parquet(EXPORT_PATH + export_name_all, index=False)
        print(f"Exported enriched DataFrame to {EXPORT_PATH + export_name_all}")
    else:
        export_name_filtered = f"df_filtered_final_{pct_grades}pct_grades_{date_stamp}.parquet"
        df_filtered_final.to_parquet(EXPORT_PATH + export_name_filtered, index=False)
        print(f"Exported enriched DataFrame to {EXPORT_PATH + export_name_filtered}")

        export_name_all = f"df_all_final_{pct_grades}pct_grades_{date_stamp}.parquet"
        df_all_final.to_parquet(EXPORT_PATH + export_name_all, index=False)
        print(f"Exported enriched DataFrame to {EXPORT_PATH + export_name_all}")
else:
    print("Enriched DataFrame not exported.")

Exported enriched DataFrame to ../../data/data_analysis_files/df_filtered_final_86pct_grades_excl_2026_09042026.parquet
Exported enriched DataFrame to ../../data/data_analysis_files/df_all_final_86pct_grades_excl_2026_09042026.parquet


The DataFrames to use are:
- `df_all_final`
- `df_filtered_final`

## Columns in DataFrames

### `df_all_final`

* abstract_ts
* Timestamp
* Author
* ID
* Publication Year
* keywords_ts
* keywords_facet
* keywords_normalized
* member_id_ss
* primary_member_id_s
* Title
* MASTER THESIS TITLE
* BY
* SUPERVISED BY
* pdf_file
* num_tot_pages
* num_cont_pages
* num_words_full
* num_words_cont
* handin_month
* num_figures
* num_tables
* num_references
* equation_count
* pdf_sha256
* total_sentences
* total_words
* unique_words
* avg_sentence_length
* avg_word_length
* lexical_diversity
* Department_new


### `df_filtered_final`

**COLUMNS RELEVANT FOR ANALYSIS:**
- **`Publication Year`**: The year of publication
- **`MASTER THESIS TITLE`**: The english title of the thesis
- **`BY`**: The author(s) in the format "lastname, name" (if multiple auhtors, they're separated wiht ";") 
- **`SUPERVISED BY`**: The supervisor(s) in the format "lastname, name" (if multiple supervisors, they're separated with ";")
- **`num_tot_pages`**: Number of total pages in .pdf file
- **`num_cont_pages`**: Number of content pages in the .pdf file (excluding appendix, references etc.)
- **`handin_month`**: The month of handin exstracted from the .pdf file. *OBS(!):* disregard the year in the stirng, and use the metric `Publication Year` for true year.
- **`num_figures`**: Number of figures in the .pdf file
- **`num_tables`**: Number of tables in the .pdf file
- **`num_references`** Number of references listed in the section regarding bibliography in the .pdf file
- **`equation_count`**: Number of equations in the .pdf file
- **`total_sentences`**: Number of sentences in main content of .pdf file
- **`total_words`**: Number of words in main content of .pdf file
- **`unique_words`**: Number of unique words in main content of .pdf file
- **`avg_sentence_length`**: Average sentence lenght of main content of .pdf file
- **`avg_word_length`**: Average word lenght of main conent of .pdf file
- **`lexical_diversity`**: Measure of the lexical diversity in the main content of .pdf file (unique_words/total_words)
- **`Department_new`**: The department of DTU from which the thesis is published
- **`grading_scientific_contribution`**: Sub grading score, (x-y)
- **`grading_methodological_rigor`**: Sub grading score, (x-y)
- **`grading_technical_implementation`**: Sub grading score, (x-y)
- **`grading_literature_review`**: Sub grading score, (x-y)
- **`grading_process_professionalism`**: Sub grading score, (x-y)
- **`grading_impact_applicability`**: Sub grading score, (x-y)
- **`grading_research_question_alignment`**: Sub grading score, (x-y)
- **`grading_total_score`**: Total assigned grading score (1-100) for the thesis by local LLM. Consistes of the scores; scientific contribution, methodological rigor, technical implementation, literature review, process professionalism, impact applicability.

***ADDED COLUMNS FOR ANALYSIS (only in this script)***
- **`num_authors`**: Number of authors for MSc Thesis, count of semicolons inn column `BY`. If the value is missing (NaN), fillna(0) treats it as 0 semicolons, resulting in 1 author.
- **`handin_month_num`** getting only the month from column `handin_month` and mapping to a number (1-12) using the calendar module for robustness.


**COLUMNS IN DataFrame, BUT NOT RELEVANT FOR ANALYSIS (or is a dublication)**
- Timestamp
- Author
- ID
- member_id_ss
- primary_member_id_s
- Title
- pdf_file
- num_words_full
- num_words_cont
- pdf_sha256
- grading_meta_attempts
- grading_meta_original_chars
- grading_meta_trimmed_at_references
- grading_meta_input_chars
- grading_meta_estimated_input_tokens
- grading_meta_was_truncated
- grading_meta_prompt_fit_attempts
- grading_meta_timeout_seconds
- grading_meta_stream_mode
